<a href="https://colab.research.google.com/github/BrionyMeng/Colab-Temp/blob/HEST-alignment/prepare_hest_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 7 · Data preparation — **RUN ONCE, BY TEACHING STAFF**

Students never run this notebook. It downloads a subset of **HEST-1k**, extracts frozen
image features and normalised expression vectors, and writes the single `.npz` file that
Parts 1 and 2 of the assignment download.

**Run it on Colab with a GPU** (Runtime → Change runtime type → T4). Expect roughly
20–40 minutes, most of it download.

---

### Before you start — two things to check

1. **HEST-1k is a gated dataset.** Go to
   <https://huggingface.co/datasets/MahmoodLab/hest>, request access, accept the terms,
   and create a read token at <https://huggingface.co/settings/tokens>. This is exactly
   why we redistribute a derived feature file rather than asking 60 students to each
   request access.
2. **Check the licence before redistributing.** HEST-1k aggregates many public studies.
   Confirm that the licence permits sharing derived features with the class, and put the
   attribution (Jaume et al., *HEST-1k*, NeurIPS 2024) in the assignment handout.

### What this produces

`hest_alignment.npz`, with:

| key | shape | meaning |
|---|---|---|
| `img_feat` | (N, 512) float32 | frozen encoder embedding of the spot's H&E patch |
| `expr` | (N, 256) float32 | log1p-normalised expression, 256 highly-variable genes |
| `gene_names` | (256,) str | which genes those are |
| `sample_id` | (N,) int32 | which HEST sample the spot came from |
| `split` | (N,) str | "train" / "val" / "test", assigned **by sample** |
| `organ`, `cancer_type` | (N,) str | sample-level labels (used in Part 2) |
| `thumbs` | (M, 64, 64, 3) uint8 | thumbnails, test spots only |
| `thumb_idx` | (M,) int32 | index of each thumbnail into the arrays above |

In [ ]:
!pip install -q "pandas<3" datasets huggingface_hub scanpy anndata h5py timm

import os
import json

import numpy as np
import h5py
import torch
import torch.nn as nn
import scanpy as sc
import anndata as ad
import pandas as pd
from huggingface_hub import login, hf_hub_download
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

# login()   # <- uncomment and paste your HF token, or set HF_TOKEN in the environment

REPO = "MahmoodLab/hest"
OUT = "hest_alignment.npz"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 123.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.8/188.8 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 126.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
device: cuda


## 1. Choose the samples

We want a handful of samples spread over a few organs, small enough that the whole thing
fits in a Colab session and the student download stays under ~60 MB.

**Set `SAMPLE_IDS` after looking at the metadata table below.** Aim for:

* 12–20 samples, 3–6 organs, so Part 2's classification task is non-trivial;
* several samples per organ, so the by-sample split can keep every organ in every split;
* `10x Visium` samples (consistent spot size and gene panel — mixing technologies here
  makes the expression side much messier than the exercise needs).

In [ ]:
meta_path = hf_hub_download(REPO, "HEST_v1_1_0.csv", repo_type="dataset")
meta = pd.read_csv(meta_path)
print(meta.shape)

MIN_SPOTS = 500           # skip tiny sections
SAMPLES_PER_ORGAN = 4     # 4 gives a clean 2 train / 1 val / 1 test per organ
N_ORGANS = 4

cand = meta[
    meta["st_technology"].astype(str).str.contains("Visium", case=False, na=False)
    & meta["species"].astype(str).str.contains("Homo sapiens", case=False, na=False)
    & meta["oncotree_code"].notna()                       # tumour samples only
    & (meta["spots_under_tissue"].fillna(0) >= MIN_SPOTS)
].copy()
print(f"{len(cand)} candidate tumour Visium samples\n")

print("=== licences present — you must be able to redistribute derived features ===")
print(cand["license"].value_counts(dropna=False).to_string(), "\n")

summary = (cand.groupby("organ")
           .agg(samples=("id", "count"),
                patients=("patient", lambda s: s.nunique()),
                median_spots=("spots_under_tissue", "median"),
                onco=("oncotree_code", lambda s: "/".join(sorted(set(s.dropna()))[:3])))
           .sort_values("patients", ascending=False))
print("=== per organ (PATIENTS is the number that matters, not samples) ===")
print(summary.to_string(), "\n")

# --- auto-suggest a selection, one sample per PATIENT ------------------------
usable = summary[summary["patients"] >= SAMPLES_PER_ORGAN].head(N_ORGANS).index.tolist()
suggested = []
for org in usable:
    sub = cand[cand["organ"] == org].sort_values("spots_under_tissue", ascending=False)
    seen = set()
    for _, r in sub.iterrows():
        if r["patient"] in seen:
            continue                      # never two samples from the same patient
        seen.add(r["patient"])
        suggested.append(r["id"])
        if len(seen) == SAMPLES_PER_ORGAN:
            break

print("=== suggested selection ===")
print(cand.set_index("id").loc[suggested][
    ["organ", "oncotree_code", "patient", "spots_under_tissue", "license"]].to_string())
print("\nSAMPLE_IDS = [")
for org in usable:
    ids = [i for i in suggested if cand.set_index("id").loc[i, "organ"] == org]
    print("    " + ", ".join(f'"{i}"' for i in ids) + f",   # {org}")
print("]")

HEST_v1_1_0.csv:   0%|          | 0.00/535k [00:00<?, ?B/s]

(1229, 28)
137 candidate tumour Visium samples

=== licences present — you must be able to redistribute derived features ===
license
CC BY-NC-ND 4.0                                   34
Creative Commons Attribution                      30
NaN                                               30
Internal                                          29
Creative Commons Attribution 4.0 International    10
CC BY-NC 4.0                                       2
CC BY 4.0                                          2 

=== per organ (PATIENTS is the number that matters, not samples) ===
            samples  patients  median_spots                onco
organ                                                          
Kidney           24        24        2893.0              SCCRCC
Brain            17        11        2743.0             EPM/GBM
Bowel            57         7        3261.0  COAD/COADREAD/READ
Bladder           6         5         924.5                BLCA
Lymph node        4         4        4992

In [ ]:
# EDIT ME. Pick ~3 samples per organ across 4-5 organs.
# Example shape of the selection (replace with real ids from the table above):
SAMPLE_IDS = [
    "INT21", "INT19", "INT15", "INT18",   # Kidney
    "TENX73", "NCBI629", "NCBI633", "NCBI638",   # Brain
    "TENX70", "MISC47", "TENX152", "ZEN47",   # Bowel
    "NCBI601", "NCBI855", "NCBI602", "NCBI600",   # Bladder
]
assert SAMPLE_IDS, "Fill in SAMPLE_IDS from the metadata table above before continuing."
sel = meta.set_index("id").loc[SAMPLE_IDS]
print(sel[["organ", "disease_state", "oncotree_code", "st_technology", "spots_under_tissue"]])

           organ disease_state oncotree_code st_technology  spots_under_tissue
id                                                                            
INT21     Kidney        Cancer        SCCRCC        Visium                4975
INT19     Kidney        Cancer        SCCRCC        Visium                4948
INT15     Kidney        Cancer        SCCRCC        Visium                4940
INT18     Kidney        Cancer        SCCRCC        Visium                4915
TENX73     Brain        Cancer           GBM        Visium               10878
NCBI629    Brain        Cancer           EPM        Visium                3620
NCBI633    Brain        Cancer           EPM        Visium                3153
NCBI638    Brain        Cancer           EPM        Visium                3060
TENX70     Bowel        Cancer      COADREAD        Visium                9080
MISC47     Bowel        Cancer          COAD        Visium                4818
TENX152    Bowel        Cancer          COAD        

## 2. Download

HEST already ships, per sample, the 224×224 patch cut around **every** ST spot
(`patches/<id>.h5`) alongside the expression matrix (`st/<id>.h5ad`). That means we never
have to open a whole-slide image — a very large saving.

In [ ]:
paths = {}
for sid in SAMPLE_IDS:
    paths[sid] = {
        "patches": hf_hub_download(REPO, f"patches/{sid}.h5", repo_type="dataset"),
        "st": hf_hub_download(REPO, f"st/{sid}.h5ad", repo_type="dataset"),
    }
    print("downloaded", sid)

patches/INT21.h5: reconstructing file:   0%|          |  0.00B /  768MB            

patches/INT21.h5: downloading bytes:           |  0.00B            

st/INT21.h5ad: reconstructing file:   0%|          |  0.00B /  191MB            

st/INT21.h5ad: downloading bytes:           |  0.00B            

downloaded INT21


...

patches/NCBI600.h5: reconstructing file:   0%|          |  0.00B /  124MB            

patches/NCBI600.h5: downloading bytes:           |  0.00B            

st/NCBI600.h5ad: reconstructing file:   0%|          |  0.00B / 21.1MB            

st/NCBI600.h5ad: downloading bytes:           |  0.00B            

downloaded NCBI600


## 3. The frozen image encoder

We use an ImageNet-pretrained **ResNet50** truncated to its 2048-d pooled feature, then
reduced to 512-d by PCA (fitted on training samples only). This keeps the student file
small and is entirely reproducible.

> **Upgrade path.** A pathology foundation model (UNI, CONCH, Phikon) gives far better
> features and would raise every number in Part 1. They are also gated on Hugging Face.
> If you have access, swap the encoder here — nothing downstream changes except that you
> must regenerate the expected-value table in the instructor notes.

In [ ]:
import torchvision

_resnet = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
_resnet.fc = nn.Identity()
ENCODER = _resnet.eval().to(DEVICE)

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


@torch.no_grad()
def encode_patches(arr, batch=128):
    """arr: (n, H, W, 3) uint8 -> (n, 2048) float32"""
    feats = []
    for i in range(0, len(arr), batch):
        x = torch.from_numpy(arr[i:i + batch]).float().permute(0, 3, 1, 2) / 255.0
        x = (x - MEAN) / STD
        feats.append(ENCODER(x.to(DEVICE)).cpu())
    return torch.cat(feats).numpy().astype(np.float32)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 165MB/s]


## 4. Read each sample, encode, and collect

The patch file stores `img` (n, 224, 224, 3) and `barcode` (n,). We match barcodes to the
AnnData rows so the pairing is exact — this is the ground-truth alignment the students'
model has to rediscover.

In [ ]:
MAX_SPOTS_PER_SAMPLE = 1200   # keeps the student download comfortably under ~70 MB

feat_2048, expr_raw, sid_arr, thumb_store = [], [], [], []
_sub_rng = np.random.default_rng(0)

for k, sid in enumerate(SAMPLE_IDS):
    with h5py.File(paths[sid]["patches"], "r") as f:
        imgs = f["img"][:]
        barcodes = np.array([b.decode() if isinstance(b, bytes) else str(b)
                             for b in f["barcode"][:].ravel()])
    adata = sc.read_h5ad(paths[sid]["st"])
    adata.var_names_make_unique()

    # keep only spots present in both
    common = np.intersect1d(barcodes, adata.obs_names.values)
    if MAX_SPOTS_PER_SAMPLE and len(common) > MAX_SPOTS_PER_SAMPLE:
        keep = np.sort(_sub_rng.choice(len(common), MAX_SPOTS_PER_SAMPLE, replace=False))
        common = common[keep]
    bmap = {b: i for i, b in enumerate(barcodes)}
    order = np.array([bmap[b] for b in common])
    imgs, adata = imgs[order], adata[common]

    feat_2048.append(encode_patches(imgs))
    expr_raw.append(adata)
    sid_arr.append(np.full(len(common), k, dtype=np.int32))
    thumb_store.append(np.stack([
        np.asarray(Image.fromarray(im).resize((64, 64), Image.BILINEAR)) for im in imgs
    ]))
    print(f"{sid}: {len(common):5d} spots, {adata.shape[1]} genes")

feat_2048 = np.concatenate(feat_2048)
sample_id = np.concatenate(sid_arr)
thumbs_all = np.concatenate(thumb_store)
print("\ntotal spots:", len(feat_2048))

INT21:  1200 spots, 17943 genes
INT19:  1200 spots, 17943 genes
INT15:  1200 spots, 17943 genes
INT18:  1200 spots, 17943 genes
TENX73:  1200 spots, 18085 genes
NCBI629:  1200 spots, 36601 genes
NCBI633:  1200 spots, 36601 genes
NCBI638:  1200 spots, 36601 genes
TENX70:  1200 spots, 18085 genes
MISC47:  1200 spots, 19327 genes
TENX152:  1200 spots, 18085 genes
ZEN47:  1200 spots, 36601 genes
NCBI601:  1200 spots, 33538 genes
NCBI855:  1200 spots, 17943 genes
NCBI602:   976 spots, 33538 genes
NCBI600:   798 spots, 33538 genes

total spots: 18574


## 5. Split by sample, then normalise

The split is assigned at the **sample** level and everything downstream — PCA, gene
selection, standardisation — is fitted on the training samples only. Getting this order
right is the whole point of the exercise's warning about batch effects; if we leak here,
the students' results are meaningless no matter what they implement.

In [ ]:
rng = np.random.default_rng(0)
organ_of = sel["organ"].astype(str).values
patient_of = sel["patient"].astype(str).values

# Split by PATIENT, stratified by organ. Splitting by *sample* is not enough: HEST
# contains several sections cut from the same patient, and a patient straddling
# train and test leaks far more than a batch effect does.
pat_df = (pd.DataFrame({"patient": patient_of, "organ": organ_of})
          .drop_duplicates("patient").reset_index(drop=True))

split_of_patient = {}
for org in pat_df["organ"].unique():
    pats = pat_df.loc[pat_df["organ"] == org, "patient"].values.copy()
    rng.shuffle(pats)
    n_tr = max(1, int(round(0.6 * len(pats))))
    n_va = 1 if len(pats) - n_tr >= 2 else 0
    for p in pats[:n_tr]:
        split_of_patient[p] = "train"
    for p in pats[n_tr:n_tr + n_va]:
        split_of_patient[p] = "val"
    for p in pats[n_tr + n_va:]:
        split_of_patient[p] = "test"

split_of_sample = np.array([split_of_patient[p] for p in patient_of])
split = split_of_sample[sample_id]
is_train = split == "train"
for sp in ["train", "val", "test"]:
    print(f"{sp:6s} {np.sum(split == sp):6d} spots  "
          f"{len(np.unique(sample_id[split == sp])):3d} samples")

train   10398 spots    9 samples
val      2400 spots    2 samples
test     5776 spots    5 samples


In [ ]:
# ---- image side: PCA 2048 -> 512, fitted on train only ----------------------
from sklearn.decomposition import PCA

pca = PCA(n_components=512, random_state=0).fit(feat_2048[is_train])
img_feat = pca.transform(feat_2048).astype(np.float32)
print(f"PCA 2048 -> 512, explained variance {pca.explained_variance_ratio_.sum():.1%}")

PCA 2048 -> 512, explained variance 97.2%


In [ ]:
# ---- expression side: harmonise genes, normalise, pick 256 HVGs on train ----
common_genes = set(expr_raw[0].var_names)
for a in expr_raw[1:]:
    common_genes &= set(a.var_names)
common_genes = sorted(common_genes)
print(f"{len(common_genes)} genes shared across all samples")

adata = ad.concat([a[:, common_genes] for a in expr_raw], axis=0, merge="same")
adata.obs["split"] = split
adata.obs["sample_id"] = sample_id

sc.pp.normalize_total(adata, target_sum=1e4)   # counts per 10k, removes depth differences
sc.pp.log1p(adata)

train_view = adata[adata.obs["split"] == "train"].copy()
sc.pp.highly_variable_genes(train_view, n_top_genes=256)
hvg = train_view.var_names[train_view.var["highly_variable"]].tolist()
assert len(hvg) == 256, len(hvg)

expr = np.asarray(adata[:, hvg].X.todense() if hasattr(adata[:, hvg].X, "todense")
                  else adata[:, hvg].X, dtype=np.float32)
print("expression matrix:", expr.shape, f"| {(expr == 0).mean():.1%} zeros")

17082 genes shared across all samples


/tmp/ipykernel_538/212039794.py:8: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  adata = ad.concat([a[:, common_genes] for a in expr_raw], axis=0, merge="same")
/tmp/ipykernel_538/212039794.py:15: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  train_view = adata[adata.obs["split"] == "train"].copy()


expression matrix: (18574, 256) | 78.7% zeros


## 6. Thumbnails (test spots only) and save

In [ ]:
test_idx = np.where(split == "test")[0]
n_thumbs = min(2000, len(test_idx))
thumb_idx = np.sort(rng.choice(test_idx, size=n_thumbs, replace=False)).astype(np.int32)
thumbs = thumbs_all[thumb_idx]

np.savez_compressed(
    OUT,
    img_feat=img_feat,
    expr=expr,
    gene_names=np.array(hvg),
    sample_id=sample_id,
    split=split,
    organ=np.array([organ_of[s] for s in sample_id]),
    cancer_type=np.array([str(sel["oncotree_code"].values[s]) for s in sample_id]),
    thumbs=thumbs,
    thumb_idx=thumb_idx,
    is_mock=np.array(False),
)
print(f"wrote {OUT}  ({os.path.getsize(OUT) / 1e6:.1f} MB)")

wrote hest_alignment.npz  (57.3 MB)


## 7. Sanity checks before you distribute this file

Run these. A green light here is what tells you the assignment will behave.

In [ ]:
d = np.load(OUT, allow_pickle=True)
ok = True


def chk(label, cond, detail=""):
    global ok
    ok &= bool(cond)
    print(f"[{'PASS' if cond else 'FAIL'}] {label}" + (f"  — {detail}" if detail else ""))


chk("no sample appears in two splits",
    all(len(np.unique(d["split"][d["sample_id"] == s])) == 1
        for s in np.unique(d["sample_id"])))
chk("no PATIENT appears in two splits",
    all(len(set(split_of_sample[patient_of == p])) == 1 for p in set(patient_of)))
chk("train and test are large enough",
    all((d["split"] == s).sum() > 1000 for s in ["train", "test"]))
if (d["split"] == "val").sum() < 500:
    print("[warn] the val split is small or empty — fine for Parts 1-3, which only use "
          "train and test, but raise SAMPLES_PER_ORGAN if you want a real val split.")
chk("every organ appears in train and test",
    set(d["organ"][d["split"] == "train"]) >= set(d["organ"][d["split"] == "test"]))
chk("no NaNs", np.isfinite(d["img_feat"]).all() and np.isfinite(d["expr"]).all())
chk("thumbnails all point at test spots", (d["split"][d["thumb_idx"]] == "test").all())
chk("file size is reasonable for a class download",
    os.path.getsize(OUT) < 120e6, f"{os.path.getsize(OUT)/1e6:.0f} MB")
chk("expression is sparse but not empty", 0.1 < (d["expr"] == 0).mean() < 0.95,
    f"{(d['expr'] == 0).mean():.1%} zeros")

print("\nREADY TO DISTRIBUTE" if ok else "\nFIX THE FAILURES ABOVE FIRST")

[PASS] no sample appears in two splits
[PASS] no PATIENT appears in two splits
[PASS] train and test are large enough
[PASS] every organ appears in train and test
[PASS] no NaNs
[PASS] thumbnails all point at test spots
[PASS] file size is reasonable for a class download  — 57 MB
[PASS] expression is sparse but not empty  — 78.7% zeros

READY TO DISTRIBUTE


In [ ]:
# Save npz to Google Drive
from google.colab import drive
drive.mount('/content/drive')
!cp {OUT} /content/drive/MyDrive/FDD3020
print("copied to your Drive")

Mounted at /content/drive
copied to your Drive


### Finally: regenerate the expected-value table

Run `part1_alignment_SOLUTION.ipynb` against this file and copy its
`part1_results.csv` into the instructor notes. Those numbers are what the TAs
compare student submissions against, so they must come from *this* file, not from
the mock data used during development.